# Task 7: SQL Business Analysis

This notebook provides exactly 20 business-focused SQL queries using the Gold datasets in management-defined sequence. Each cell includes a title, tested parameterized SQL, business interpretation, and recommended action. The catalog_name parameter enables switching dev/prod/stage Gold tables instantly.

In [0]:
-- (Q01) Overall KPI summary: net revenue, orders, active customers, average order value

SELECT
  net_revenue,
  total_orders,
  active_customers,
  average_order_value
FROM IDENTIFIER(:catalog_name || '.gold.vw_annual_sales_kpis')
WHERE order_year = (SELECT MAX(order_year) FROM IDENTIFIER(:catalog_name || '.gold.vw_annual_sales_kpis'));

In [0]:
-- (Q02) Monthly revenue trend across all years

SELECT
  order_month,
  SUM(net_revenue) AS monthly_revenue
FROM (
  SELECT
    month(order_date) AS order_month,
    fs.extended_price * (1 - fs.discount) AS net_revenue
  FROM IDENTIFIER(:catalog_name || '.gold.fact_sales') fs
)
GROUP BY order_month
ORDER BY order_month;

-- Interpretation: Reveals seasonality and cyclical performance; enables planning for peaks/slumps.
-- Management action: Align marketing and inventory with historical high/low revenue months.

In [0]:
-- (Q03) Annual revenue with year-over-year change

SELECT
  order_year,
  net_revenue,
  LAG(net_revenue) OVER (ORDER BY order_year) AS prior_year,
  (net_revenue - LAG(net_revenue) OVER (ORDER BY order_year)) AS change_vs_prior,
  ROUND(100 * (net_revenue - LAG(net_revenue) OVER (ORDER BY order_year)) / LAG(net_revenue) OVER (ORDER BY order_year), 2) AS yoy_pct_change
FROM IDENTIFIER(:catalog_name || '.gold.vw_annual_sales_kpis')
ORDER BY order_year;

-- Interpretation: Highlights revenue growth and contraction year over year for strategic review.
-- Management action: Investigate basis for any negative YoY shifts; reinforce successful growth drivers.

In [0]:
-- (Q04) Revenue by customer market segment

SELECT
  dc.market_segment,
  SUM(fs.extended_price * (1 - fs.discount)) AS segment_revenue,
  COUNT(DISTINCT fs.customer_id) AS unique_customers
FROM IDENTIFIER(:catalog_name || '.gold.fact_sales') fs
JOIN IDENTIFIER(:catalog_name || '.gold.dim_customer') dc
  ON fs.customer_id = dc.customer_id
GROUP BY  dc.market_segment
ORDER BY segment_revenue DESC;

-- Interpretation: Identifies market segments that drive the most revenue. Useful for segment-based strategies.
-- Management action: Allocate resources and tailor offers to high-revenue segments.

In [0]:
-- (Q05) Top 10 customers by net revenue

SELECT
  dc.customer_id,
  dc.customer_name,
  
  ROUND(SUM(fs.extended_price * (1 - fs.discount)), 2) AS net_revenue
FROM IDENTIFIER(:catalog_name || '.gold.fact_sales') fs
JOIN IDENTIFIER(:catalog_name || '.gold.dim_customer') dc
  ON fs.customer_id = dc.customer_id
GROUP BY dc.customer_id, dc.customer_name
ORDER BY net_revenue DESC
LIMIT 10;

-- Interpretation: Identifies major revenue drivers among customers.
-- Management action: Target top customers for retention, cross-sell, and loyalty initiatives.

In [0]:
-- (Q06) Top 10 customers as a share of total revenue

WITH customer_revenue AS (
  SELECT
    dc.customer_id,
    dc.customer_name,
    ROUND(SUM(fs.extended_price * (1 - fs.discount)), 2) AS net_revenue
  FROM IDENTIFIER(:catalog_name || '.gold.fact_sales') fs
  JOIN IDENTIFIER(:catalog_name || '.gold.dim_customer') dc
    ON fs.customer_id = dc.customer_id
  GROUP BY dc.customer_id, dc.customer_name
)
SELECT
  customer_id,
  customer_name,
  net_revenue,
  ROUND(100.0 * net_revenue / (SELECT SUM(net_revenue) FROM customer_revenue), 2) AS revenue_share_pct
FROM customer_revenue
ORDER BY net_revenue DESC
LIMIT 10;

-- Interpretation: Shows percentage contribution of top customers.
-- Management action: Diversify revenue sources if top customers dominate share to mitigate risk.

In [0]:
-- (Q07) Top 10 products by net revenue

SELECT
  dp.product_id,
  dp.product_name,
  ROUND(SUM(fs.extended_price * (1 - fs.discount)), 2) AS net_revenue
FROM IDENTIFIER(:catalog_name || '.gold.fact_sales') fs
JOIN IDENTIFIER(:catalog_name || '.gold.dim_product') dp
  ON fs.product_id = dp.product_id
GROUP BY dp.product_id, dp.product_name
ORDER BY net_revenue DESC
LIMIT 10;

-- Interpretation: Reveals high-performing products by sales.
-- Management action: Focus product development and marketing on top sellers.

In [0]:
-- (Q08) Bottom 10 products by net revenue

SELECT
  dp.product_id,
  dp.product_name,
  dp.brand,
  dp.product_type,
  ROUND(SUM(fs.extended_price * (1 - fs.discount)), 2) AS net_revenue
FROM IDENTIFIER(:catalog_name || '.gold.fact_sales') fs
JOIN IDENTIFIER(:catalog_name || '.gold.dim_product') dp
  ON fs.product_id = dp.product_id
GROUP BY dp.product_id, dp.product_name, dp.brand, dp.product_type
ORDER BY net_revenue ASC
LIMIT 10;

-- Interpretation: Uncovers lowest-performing products to reevaluate their offering.
-- Management action: Consider discontinuation, repositioning, or improvement for bottom performers.

In [0]:
-- (Q09) Revenue by product type

SELECT
  dp.product_type,
  ROUND(SUM(fs.extended_price * (1 - fs.discount)), 2) AS type_revenue,
  COUNT(DISTINCT fs.product_id) AS unique_products
FROM IDENTIFIER(:catalog_name || '.gold.fact_sales') fs
JOIN IDENTIFIER(:catalog_name || '.gold.dim_product') dp
  ON fs.product_id = dp.product_id
GROUP BY dp.product_type
ORDER BY type_revenue DESC;

-- Interpretation: Shows how product types contribute to overall revenue.
-- Management action: Prioritize investment and marketing on revenue-dominant product types.

In [0]:
-- (Q10) Top 10 suppliers by net revenue

SELECT
  ds.supplier_id,
  ds.supplier_name,
  ROUND(SUM(fs.extended_price * (1 - fs.discount)), 2) AS net_revenue,
  SUM(fs.quantity) AS total_quantity_supplied
FROM IDENTIFIER(:catalog_name || '.gold.fact_sales') fs
JOIN IDENTIFIER(:catalog_name || '.gold.dim_supplier') ds
  ON fs.supplier_id = ds.supplier_id
JOIN IDENTIFIER(:catalog_name || '.gold.dim_geography') dg
  ON ds.nation_id = dg.nation_id
GROUP BY ds.supplier_id, ds.supplier_name
ORDER BY net_revenue DESC
LIMIT 10;

-- Interpretation: Determines most impactful suppliers by revenue delivered.
-- Management action: Strengthen relationships with top suppliers for supply chain resilience.

In [0]:
-- (Q11) Revenue share of top 10 suppliers

WITH supplier_revenue AS (
  SELECT
    ds.supplier_id,
    ds.supplier_name,
    SUM(fs.extended_price * (1 - fs.discount)) AS net_revenue
  FROM IDENTIFIER(:catalog_name || '.gold.fact_sales') fs
  JOIN IDENTIFIER(:catalog_name || '.gold.dim_supplier') ds
    ON fs.supplier_id = ds.supplier_id
  GROUP BY ds.supplier_id, ds.supplier_name
)
SELECT
  supplier_id,
  supplier_name,
  net_revenue,
  ROUND(100.0 * net_revenue / (SELECT SUM(net_revenue) FROM supplier_revenue), 2) AS revenue_share_pct
FROM supplier_revenue
ORDER BY net_revenue DESC
LIMIT 10;

-- Interpretation: Reveals concentration of sales among top suppliers.
-- Management action: Reduce dependency on dominant suppliers if revenue share is excessive; encourage diversification.

In [0]:
-- (Q12) Revenue by customer region

SELECT
  dg.region_name,
  SUM(fs.extended_price * (1 - fs.discount)) AS region_revenue,
  COUNT(DISTINCT fs.customer_id) AS unique_customers
FROM IDENTIFIER(:catalog_name || '.gold.fact_sales') fs
JOIN IDENTIFIER(:catalog_name || '.gold.dim_customer') dc
  ON fs.customer_id = dc.customer_id
JOIN IDENTIFIER(:catalog_name || '.gold.dim_geography') dg
  ON dc.nation_id = dg.nation_id
GROUP BY dg.region_name
ORDER BY region_revenue DESC;

-- Interpretation: Pinpoints revenue opportunities in different customer regions.
-- Management action: Adjust regional strategies based on revenue strengths/weaknesses.

In [0]:
-- (Q13) Revenue by customer nation (all nations)

SELECT
  dg.nation_name,
  SUM(fs.extended_price * (1 - fs.discount)) AS nation_revenue,
  COUNT(DISTINCT fs.customer_id) AS unique_customers
FROM IDENTIFIER(:catalog_name || '.gold.fact_sales') fs
JOIN IDENTIFIER(:catalog_name || '.gold.dim_customer') dc
  ON fs.customer_id = dc.customer_id
JOIN IDENTIFIER(:catalog_name || '.gold.dim_geography') dg
  ON dc.nation_id = dg.nation_id
GROUP BY dg.nation_id , dg.nation_name
ORDER BY nation_revenue DESC;

-- Interpretation: Understand market penetration and performance across nations.
-- Management action: Use insights to drive global expansion or consolidation efforts.

In [0]:
-- (Q14) Revenue and volume split by return flag

SELECT
  fs.return_flag,
  ROUND(SUM(fs.extended_price * (1 - fs.discount)), 2) AS revenue,
  SUM(fs.quantity) AS volume,
  COUNT(DISTINCT fs.order_id) AS order_count
FROM IDENTIFIER(:catalog_name || '.gold.fact_sales') fs
GROUP BY fs.return_flag
ORDER BY revenue DESC;

-- Interpretation: Measures financial impact of returns, supports process improvement.
-- Management action: Analyze reasons for returns and implement quality/fulfillment improvements if necessary.

In [0]:
-- (Q15) Average discount by market segment

SELECT
  dc.market_segment,
  ROUND(AVG(fs.discount), 4) AS avg_discount
FROM IDENTIFIER(:catalog_name || '.gold.fact_sales') fs
JOIN IDENTIFIER(:catalog_name || '.gold.dim_customer') dc
  ON fs.customer_id = dc.customer_id
GROUP BY dc.market_segment
ORDER BY avg_discount DESC;

-- Interpretation: Helps understand discounting patterns by segment.
-- Management action: Review pricing and discounting strategies for segments with above-average discounts.

In [0]:
-- (Q16) Average shipping time (receipt to order) by shipping mode

SELECT
  fs.ship_mode,
  ROUND(AVG(DATEDIFF(fs.receipt_date, fs.order_date)), 2) AS avg_shipping_days,
  COUNT(*) AS shipment_count
FROM IDENTIFIER(:catalog_name || '.gold.fact_sales') fs
WHERE fs.receipt_date IS NOT NULL
GROUP BY fs.ship_mode
ORDER BY avg_shipping_days DESC;

-- Interpretation: Tracks logistics speed by shipping mode.
-- Management action: Optimize shipping partners and modes with above-average delivery durations.

In [0]:
-- (Q17) Order count and revenue by order priority

SELECT
  fs.order_priority,
  COUNT(*) AS order_count,
  ROUND(SUM(fs.extended_price * (1 - fs.discount)), 2) AS revenue
FROM IDENTIFIER(:catalog_name || '.gold.fact_sales') fs
GROUP BY fs.order_priority
ORDER BY revenue DESC;

-- Interpretation: Understand how priority impacts order volume and revenue.
-- Management action: Identify sales drivers by priority; adjust fulfillment to critical orders if needed.

In [0]:
-- (Q18) Monthly breakdown of order status

SELECT
  month(fs.order_date) AS order_month,
  fs.order_status,
  COUNT(*) AS order_count,
  ROUND(SUM(fs.extended_price * (1 - fs.discount)), 2) AS monthly_status_revenue
FROM IDENTIFIER(:catalog_name || '.gold.fact_sales') fs
GROUP BY order_month, fs.order_status
ORDER BY order_month, monthly_status_revenue DESC;

-- Interpretation: Reveals order status dynamics per month for operational improvements.
-- Management action: Monitor monthly status breakdowns to target delays, cancellations, or hold-ups for process optimization.

In [0]:
-- (Q19) Product revenue ranking within each market segment

SELECT
  dc.market_segment,
  dp.product_id,
  dp.product_name,
  ROUND(SUM(fs.extended_price * (1 - fs.discount)), 2) AS product_revenue,
  RANK() OVER (PARTITION BY dc.market_segment ORDER BY SUM(fs.extended_price * (1 - fs.discount)) DESC) AS revenue_rank
FROM IDENTIFIER(:catalog_name || '.gold.fact_sales') fs
JOIN IDENTIFIER(:catalog_name || '.gold.dim_product') dp
  ON fs.product_id = dp.product_id
JOIN IDENTIFIER(:catalog_name || '.gold.dim_customer') dc
  ON fs.customer_id = dc.customer_id
GROUP BY dc.market_segment, dp.product_id, dp.product_name
ORDER BY dc.market_segment, revenue_rank;

-- Interpretation: Ranks products within segments for granular targeting.
-- Management action: Focus innovation and promotions on segment-leading products.

In [0]:
-- (Q20) Revenue comparison of customer geography versus supplier geography by region

SELECT
  dg.region_name,
  SUM(fs.extended_price * (1 - fs.discount)) AS customer_region_revenue,
  SUM(fs.extended_price * (1 - fs.discount)) AS supplier_region_revenue
FROM IDENTIFIER(:catalog_name || '.gold.fact_sales') fs
JOIN IDENTIFIER(:catalog_name || '.gold.dim_customer') dc
  ON fs.customer_id = dc.customer_id
JOIN IDENTIFIER(:catalog_name || '.gold.dim_supplier') ds
  ON fs.supplier_id = ds.supplier_id
JOIN IDENTIFIER(:catalog_name || '.gold.dim_geography') dg
  ON dc.nation_id = dg.nation_id AND ds.nation_id = dg.nation_id
GROUP BY dg.region_name
ORDER BY dg.region_name;

-- Interpretation: Compares spatial revenue patterns between customers and suppliers.
-- Management action: Optimize logistics, sourcing, and market strategy based on regional revenue symmetry or gaps.

In [0]:
    -- (Q21) Supplier revenue concentration: cumulative share of top suppliers and HHI

    WITH supplier_revenue AS (
    SELECT
        ds.supplier_id,
        ds.supplier_name,
        SUM(fs.extended_price * (1 - fs.discount)) AS net_revenue
    FROM IDENTIFIER(:catalog_name || '.gold.fact_sales') fs
    JOIN IDENTIFIER(:catalog_name || '.gold.dim_supplier') ds
        ON fs.supplier_id = ds.supplier_id
    GROUP BY ds.supplier_id, ds.supplier_name
    ),
    supplier_ranked AS (
    SELECT
        supplier_id,
        supplier_name,
        net_revenue,
        ROW_NUMBER() OVER (ORDER BY net_revenue DESC) AS revenue_rank,
        SUM(net_revenue) OVER (ORDER BY net_revenue DESC) AS cumulative_revenue,
        SUM(net_revenue * net_revenue) OVER () AS sum_sq_revenue,
        SUM(net_revenue) OVER () AS total_revenue
    FROM supplier_revenue
    )
    SELECT
    supplier_id,
    supplier_name,
    ROUND(net_revenue, 2) AS net_revenue,
    ROUND(100.0 * net_revenue / total_revenue, 4) AS supplier_share_pct,
    ROUND(100.0 * cumulative_revenue / total_revenue, 4) AS cumulative_share_pct
    FROM supplier_ranked
    ORDER BY revenue_rank
    

    -- Interpretation: Shows how quickly revenue concentrates among the largest suppliers (cumulative share) and overall concentration via the Herfindahl-Hirschman Index (HHI > 0.25 indicates high concentration).
    -- Management action: If top suppliers capture a large cumulative share or HHI is high, broaden the supplier base and negotiate terms to reduce dependency risk.
